# 02 · Model and embeddings

Loads the trained encoder, computes (or loads from the **embedding cache**)
the full-dataset embeddings for every modality, then inspects the shared
embedding space: L2 normalisation, class separation and the PCA/UMAP view.

The 128-D embeddings are L2-normalised and live in a *shared* space, so images
with the same land-cover class end up close together regardless of sensor.

In [ ]:
import os, sys
PROJ = os.path.dirname(os.getcwd()) if os.path.split(os.getcwd())[1] == 'notebooks' else os.getcwd()
if PROJ not in sys.path: sys.path.insert(0, PROJ)
nb_dir = os.path.join(PROJ, 'notebooks')
if nb_dir not in sys.path: sys.path.insert(0, nb_dir)
import utils
print('project root:', PROJ)


In [ ]:
import numpy as np
P = utils.load_pipeline('configs/default.yaml')   # model + engine + persistence
engine, cfg = P['engine'], P['cfg']
print('modalities:', cfg['modalities'], ' embedding dim:', cfg['model']['embedding_dim'])
print('config hash:', engine.config_hash)

In [ ]:
# Ensure full embeddings are cached (hits disk on warm starts).
emb = {}
for m in cfg['modalities']:
    emb[m], _ = engine.cache_full_embeddings(m)
    print(f'{m:<14} embeddings shape={emb[m].shape}  (L2 norm per row ≈ {np.linalg.norm(emb[m], axis=1).mean():.4f})')

### Embedding space by modality (PCA to 2-D, coloured by class)

In [ ]:
for m in cfg['modalities']:
    utils.plot_embedding_2d(emb[m], P['labels'], P['class_names'], method='pca',
                            title=f'Embedding space · {m} (PCA)');

### Cross-modal alignment
If the encoder has truly aligned the modalities, the same patch should be near its
counterpart from another sensor. We measure the **self-similarity**: for each patch,
the rank of its own embedding from a *different* modality within the gallery.

In [ ]:
from src.retrieval.index import build_index_from_embeddings
import time
o, s = 'optical', 'sar'
idx = build_index_from_embeddings(emb[o])   # search SAR queries in the OPTICAL gallery
t0 = time.perf_counter()
_, ids = idx.search(emb[s], 20)
rank = np.where(ids == np.arange(ids.shape[0])[:, None])[1].astype(float)
rank = np.where(rank == 0, rank, np.nan)   # keep only patches whose SAR twin is found
found = np.where(ids == np.arange(ids.shape[0])[:, None])[0]
r = [np.where(ids[i] == i)[0][0] if np.any(ids[i] == i) else np.nan for i in range(len(ids))]
valid = [x for x in r if not np.isnan(x)]
print(f'SAR->optical self-similarity: twin found in top-20 for {len(valid)}/{len(ids)} patches')
print(f'  median self rank: {np.median(valid):.0f}')
print(f'  search time (full gallery): {(time.perf_counter()-t0)*1000/len(ids)*1000:.0f} µs/query')

### Class separation
A simple cluster-quality score: the ratio of mean *between*-class distance to mean
*within*-class distance (higher ⇒ better separated classes).

In [ ]:
from sklearn.metrics import pairwise_distances
lab = P['labels']
d = pairwise_distances(emb['optical'])
within, between = [], []
for a in range(len(emb['optical'])):
    same = lab == lab[a]
    within.append(d[a, same].mean()); between.append(d[a, ~same].mean())
print(f'within-class mean dist : {np.mean(within):.3f}')
print(f'between-class mean dist: {np.mean(between):.3f}')
print(f'separation ratio       : {np.mean(between)/np.mean(within):.2f}  (larger = better)')

---
Next: [03_interactive_retrieval.ipynb](03_interactive_retrieval.ipynb) turns the
embedding space into ranked retrieval.